This code will tokenize Pile (https://huggingface.co/datasets/monology/pile-uncopyrighted) and Lymsys (https://huggingface.co/datasets/lmsys/lmsys-chat-1m) - actually using (https://huggingface.co/datasets/science-of-finetuning/lmsys-chat-1m-chat-formatted) datasets for Qwen - (For ~400 mil tokens) and upload them to HF for training 

In [ ]:
from itertools import islice
from datasets import load_dataset, concatenate_datasets, Dataset
from tqdm import tqdm

# streaming datasets
pile_streamed  = load_dataset("monology/pile-uncopyrighted",
                              split="train", streaming=True)
lmsys_streamed = load_dataset("science-of-finetuning/lmsys-chat-1m-chat-formatted",
                              split="train", streaming=True)

def process_pile(ex): return {"text": ex["text"], "dataset": "pile"}
def process_lmsys(ex): return {"text": ex["text"], "dataset": "lmsys"}

pile_iter   = map(process_pile,  pile_streamed)
lmsys_iter  = map(process_lmsys, lmsys_streamed)

# Taking 500_000 samples
from itertools import islice

pile_list  = list(tqdm(islice(pile_iter,  500_000),
                       total=500_000, desc="Loading Pile"))
lmsys_list = list(tqdm(islice(lmsys_iter, 500_000),
                       total=500_000, desc="Loading LMSYS"))

pile_ds, lmsys_ds = map(Dataset.from_list, (pile_list, lmsys_list)) # Crash


Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading LMSYS: 100%|██████████| 500000/500000 [12:56<00:00, 644.12it/s] 


: 

In [ ]:
# Prep Llama3.2-1B dataset with chat template and BOS tokens (BOS token can be added by sae_lens PretokenizeRunner)

# for each line in lmsys_processed replace <start_of_turn> <end_of_turn> with their counterparts in Llama 3.2
# Tokenizer for Llama https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct/blob/main/tokenizer_config.json 
# Info about chat template https://www.llama.com/docs/model-cards-and-prompt-formats/meta-llama-3/?utm_source=chatgpt.com

from transformers import AutoTokenizer
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)

BEGIN   = tokenizer.bos_token or "<|begin_of_text|>" # BOS - only once per batch
SOT   = "<|start_header_id|>" # header is system, user or assistant, so the header tokens sandwitch these tokens
END_H     = "<|end_header_id|>" # Needs to placed after text "user" or model, make some logic that places it after <|start_header_id|>, so we don't accidently place it
                              # somwhere where a user mentions "user" or "model" # In Lmysys assitant = model, user = user
EOT     = "<|eot_id|>" # This token signifies the end of turn for assitant or user, place before <|start_header_id|

TOKEN_MAP = {
    "<start_of_turn>": SOT,
    "<end_of_turn>":   EOT,
}

# Replacement logic
import re
def format_lmsys_llama3(example):
    text = example["text"]

    # 1. Replace <start_of_turn> and <end_of_turn> with mapped tokens
    for old, new in TOKEN_MAP.items():
        text = text.replace(old, new)

    # 2. Add END_H after header labels following start_header_id
    text = re.sub(rf"{re.escape(SOT)}(system|user|assistant)", rf"{SOT}\1{END_H}", text)

    # 3. Ensure EOT comes before each new header
    text = re.sub(rf"({END_H}\n?)", rf"\1{EOT}\n", text)

    # 4. Add BOS token at the beginning if not present
    if not text.startswith(BEGIN):
        text = BEGIN + text

    return {"text": text, "dataset": "lmsys"}


# lmsys_ds = map(format_lmsys_llama3, lmsys_streamed)

# Get one sample from the LMSYS stream
raw_sample = next(iter(lmsys_streamed))

# Apply the Llama3.2 formatting logic
formatted_sample = format_lmsys_llama3(raw_sample)

# Print the result
print("Formatted LMSYS sample with Llama 3.2 tokens:\n")
print(formatted_sample["text"])


In [ ]:
# Combine the datasets
combined_dataset = concatenate_datasets([pile_ds, lmsys_ds])

In [ ]:
# Upload to Hugging Face Hub
hf_repo_id = "AndrisWillow/pile-lmsys-mix-1m-Llama3.2_chat_format"
combined_dataset.push_to_hub(hf_repo_id)

In [ ]:
from transformers import AutoTokenizer
from sae_lens import PretokenizeRunner, PretokenizeRunnerConfig

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")

# Tokenizer for Qwen https://huggingface.co/Qwen/Qwen2.5-0.5B/blob/main/tokenizer_config.json # Does not have BOS

cfg = PretokenizeRunnerConfig(
    tokenizer_name="Qwen/Qwen2.5-0.5B",
    dataset_path="AndrisWillow/pile-lmsys_qwen_format-mix-1m",
    shuffle=True,
    num_proc=32,
    context_size=1024,
    begin_batch_token=None,  # We handle all of the formating before
    begin_sequence_token=None, 
    sequence_separator_token=None,
    hf_repo_id="AndrisWillow/Pile-Lmsys-1m-tokenized-1024-Qwen2.5-WTemplTok",
)

dataset = PretokenizeRunner(cfg).run()
